In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
files = os.listdir(
    "/content/drive/MyDrive/Colab Notebooks/output/statistics"
)
files

['sampled_all_systems.csv',
 'concat_flaml.csv',
 'concat_ag.csv',
 'result_monotonic_llama.csv',
 'result_monotonic_mistral.csv',
 'llama.csv',
 'mistral.csv',
 'result_equal_monotonic_ag_equal.csv',
 'result_monotonic_ag_equal.csv',
 'result_equal_monotonic_flaml_equal.csv',
 'result_monotonic_flaml_equal.csv',
 'result_equal_monotonic_rf_equal.csv',
 'result_monotonic_rf_equal.csv',
 'old_concat_rf.csv',
 'old_hpo_all_systems.csv',
 'concat_rf.csv',
 'hpo_all_systems.csv']

# Random Forest with Random Search

In [ ]:
system = "rf"
system_map = {
    "rf": "Random Forest with Random Search",
    "ag": "AutoGluon",
    "flaml": "FLAML",
}

## HPO

In [ ]:
hpo = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/output/statistics/hpo_all_systems.csv",
    sep=";"
)
hpo = hpo[hpo["automl_system"] == system_map[system]]
hpo["are_concat_pred_equal"] = hpo.apply(
    lambda row: row["trained_dataset"] == row["predicted_dataset"],
    axis=1
)
equal_hpo = hpo[hpo["are_concat_pred_equal"]]
diff_hpo = hpo[~(hpo["are_concat_pred_equal"])]

In [ ]:
# Looking at one HPO equal dataset example
equal_hpo.query("trained_dataset == 'ua_01'")

,trained_dataset,predicted_dataset,automl_system,method,percentage,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels,are_concat_pred_equal
415,ua_01,ua_01,Random Forest with Random Search,gmm_random,100,"['7', '5', '3', '1']",[],True,0.55838,0.55838,"['7', '5', '10', '1', '3']",['10'],True


In [ ]:
# Looking at diff HPO equal dataset example
diff_hpo.query("trained_dataset == 'ua_01'")

,trained_dataset,predicted_dataset,automl_system,method,percentage,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels,are_concat_pred_equal
411,ua_01,ua_03,Random Forest with Random Search,gmm_random,100,"['7', '5', '3', '1']",[],True,0.666016,0.666016,"['7', '10', '5', '1', '3']",['10'],False
412,ua_01,ua_04,Random Forest with Random Search,gmm_random,100,"['7', '5', '3', '1']",[],True,0.200040,0.200040,"['7', '10', '5', '1', '3']",['10'],False
413,ua_01,ua_05,Random Forest with Random Search,gmm_random,100,"['7', '5', '3', '1']",[],True,0.776119,0.776119,"['7', '10', '5', '1', '3']",['10'],False
414,ua_01,ua_06,Random Forest with Random Search,gmm_random,100,"['7', '5', '3', '1']",[],True,0.442095,0.442095,"['7', '5', '10', '1', '3']",['10'],False


In [ ]:
a = equal_hpo["trained_dataset"].unique()
a.sort()
len(a), a

(49,
 array(['ed_01', 'ed_02', 'ed_03', 'ed_04', 'ed_05', 'fn_01', 'fn_02',
        'fn_03', 'fn_04', 'fn_05', 'fn_06', 'fn_07', 'fn_08', 'fn_09',
        'fn_10', 'hs_01', 'hs_02', 'hs_03', 'hs_04', 'hs_05', 'hs_06',
        'hs_07', 'hs_08', 'hs_09', 'hs_10', 'hs_11', 'hs_12', 'hs_13',
        'hs_14', 'hs_15', 'pc_01', 'pc_02', 'pc_03', 'pc_04', 'pc_05',
        'pc_06', 'pc_07', 'pc_08', 'pc_09', 'sd_01', 'sd_02', 'sd_03',
        'sd_04', 'ua_01', 'ua_02', 'ua_03', 'ua_04', 'ua_05', 'ua_06'],
       dtype=object))

## Concat

In [ ]:
automl_concat_df = pd.read_csv(
    f"/content/drive/MyDrive/Colab Notebooks/output/statistics/concat_{system}.csv",
    sep=";"
)

adapt_mask = automl_concat_df["concatenated_dataset"] == automl_concat_df["predicted_dataset"]
aug_mask = automl_concat_df["trained_dataset"] == automl_concat_df["predicted_dataset"]

equal_concat_df = automl_concat_df.loc[adapt_mask & (~aug_mask)]
diff_concat_df = automl_concat_df.loc[(~adapt_mask) & (~aug_mask)]

In [ ]:
# Looking at one equal concat dataset example
equal_concat_df.query("trained_dataset == 'ua_01'")

,trained_dataset,predicted_dataset,concatenated_dataset,automl_system,method,percentage,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels
1650,ua_01,ua_03,ua_03,Random Forest with Random Search,gmm_random,25,"['1', '5', '3', '7']",[],True,0.666016,0.666016,"['10', '7', '5', '1', '3']",['10']
1655,ua_01,ua_03,ua_03,Random Forest with Random Search,gmm_random,50,"['1', '5', '3', '7']",[],True,0.666016,0.666016,"['10', '7', '5', '1', '3']",['10']
1660,ua_01,ua_03,ua_03,Random Forest with Random Search,gmm_random,75,"['1', '5', '3', '7']",[],True,0.666016,0.666016,"['10', '7', '5', '1', '3']",['10']
1666,ua_01,ua_04,ua_04,Random Forest with Random Search,gmm_random,25,"['1', '5', '3', '7']",[],True,0.200040,0.200040,"['10', '7', '5', '1', '3']",['10']
1671,ua_01,ua_04,ua_04,Random Forest with Random Search,gmm_random,50,"['1', '5', '3', '7']",[],True,0.200040,0.200040,"['10', '7', '5', '1', '3']",['10']
1676,ua_01,ua_04,ua_04,Random Forest with Random Search,gmm_random,75,"['1', '5', '3', '7']",[],True,0.200040,0.200040,"['10', '7', '5', '1', '3']",['10']
1682,ua_01,ua_05,ua_05,Random Forest with Random Search,gmm_random,25,"['1', '5', '3', '7']",[],True,0.776119,0.776119,"['10', '7', '5', '1', '3']",['10']
1687,ua_01,ua_05,ua_05,Random Forest with Random Search,gmm_random,50,"['1', '5', '3', '7']",[],True,0.776119,0.776119,"['10', '7', '5', '1', '3']",['10']
1692,ua_01,ua_05,ua_05,Random Forest with Random Search,gmm_random,75,"['1', '5', '3', '7']",[],True,0.776119,0.776119,"['10', '7', '5', '1', '3']",['10']
1698,ua_01,ua_06,ua_06,Random Forest with Random Search,gmm_random,25,"['1', '5', '3', '7']",[],True,0.442095,0.442095,"['10', '7', '5', '1', '3']",['10']


In [ ]:
# Looking at one diff concat dataset example
diff_concat_df.query("trained_dataset == 'ua_01'")

,trained_dataset,predicted_dataset,concatenated_dataset,automl_system,method,percentage,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels
1651,ua_01,ua_04,ua_03,Random Forest with Random Search,gmm_random,25,"['1', '5', '3', '7']",[],True,0.200040,0.200040,"['10', '7', '5', '1', '3']",['10']
1652,ua_01,ua_05,ua_03,Random Forest with Random Search,gmm_random,25,"['1', '5', '3', '7']",[],True,0.776119,0.776119,"['10', '7', '5', '1', '3']",['10']
1653,ua_01,ua_06,ua_03,Random Forest with Random Search,gmm_random,25,"['1', '5', '3', '7']",[],True,0.442095,0.442095,"['10', '7', '5', '1', '3']",['10']
1656,ua_01,ua_04,ua_03,Random Forest with Random Search,gmm_random,50,"['1', '5', '3', '7']",[],True,0.200040,0.200040,"['10', '7', '5', '1', '3']",['10']
1657,ua_01,ua_05,ua_03,Random Forest with Random Search,gmm_random,50,"['1', '5', '3', '7']",[],True,0.776119,0.776119,"['10', '7', '5', '1', '3']",['10']
1658,ua_01,ua_06,ua_03,Random Forest with Random Search,gmm_random,50,"['1', '5', '3', '7']",[],True,0.442095,0.442095,"['10', '7', '5', '1', '3']",['10']
1661,ua_01,ua_04,ua_03,Random Forest with Random Search,gmm_random,75,"['1', '5', '3', '7']",[],True,0.200040,0.200040,"['10', '7', '5', '1', '3']",['10']
1662,ua_01,ua_05,ua_03,Random Forest with Random Search,gmm_random,75,"['1', '5', '3', '7']",[],True,0.776119,0.776119,"['10', '7', '5', '1', '3']",['10']
1663,ua_01,ua_06,ua_03,Random Forest with Random Search,gmm_random,75,"['1', '5', '3', '7']",[],True,0.442095,0.442095,"['10', '7', '5', '1', '3']",['10']
1665,ua_01,ua_03,ua_04,Random Forest with Random Search,gmm_random,25,"['1', '5', '3', '7']",[],True,0.666016,0.666016,"['10', '7', '5', '1', '3']",['10']


## Equal

### 25, 50, 75

In [ ]:
grouped_equal_concat_df = (
    equal_concat_df.groupby(
        [
            "predicted_dataset",
            "automl_system",
            "method",
            "percentage"
        ]
    )
    .agg(
        {
            "hold_out_f1_score": "mean"
        }
    )
)
grouped_equal_concat_df = grouped_equal_concat_df.reset_index()
grouped_equal_concat_df

,predicted_dataset,automl_system,method,percentage,hold_out_f1_score
0,ed_01,Random Forest with Random Search,gmm_random,25,0.000000
1,ed_01,Random Forest with Random Search,gmm_random,50,0.000000
2,ed_01,Random Forest with Random Search,gmm_random,75,0.000000
3,ed_02,Random Forest with Random Search,gmm_random,25,0.425741
4,ed_02,Random Forest with Random Search,gmm_random,50,0.442578
...,...,...,...,...,...
100,ua_05,Random Forest with Random Search,gmm_random,50,0.651741
101,ua_05,Random Forest with Random Search,gmm_random,75,0.662687
102,ua_06,Random Forest with Random Search,gmm_random,25,0.443624
103,ua_06,Random Forest with Random Search,gmm_random,50,0.443266


### 0%

In [ ]:
grouped_zero_pct_hpo = (
    diff_hpo.groupby(
        [
            "predicted_dataset",
            "automl_system",
            "method"
        ]
    )
    .agg(
        {
            "hold_out_f1_score": "mean"
        }
    )
)
grouped_zero_pct_hpo = grouped_zero_pct_hpo.reset_index()
grouped_zero_pct_hpo["percentage"] = 0
grouped_zero_pct_hpo

,predicted_dataset,automl_system,method,hold_out_f1_score,percentage
0,ed_01,Random Forest with Random Search,gmm_random,0.000000,0
1,ed_02,Random Forest with Random Search,gmm_random,0.000000,0
2,fn_02,Random Forest with Random Search,gmm_random,0.304167,0
3,fn_05,Random Forest with Random Search,gmm_random,0.377322,0
4,fn_06,Random Forest with Random Search,gmm_random,0.292827,0
5,fn_07,Random Forest with Random Search,gmm_random,0.369318,0
6,fn_09,Random Forest with Random Search,gmm_random,0.261526,0
7,fn_10,Random Forest with Random Search,gmm_random,0.108333,0
8,hs_01,Random Forest with Random Search,gmm_random,0.448450,0
9,hs_02,Random Forest with Random Search,gmm_random,0.112956,0


### 100%

In [ ]:
# There is only one point per dataset since trained = predicted, but we are still grouping by and determining the mean for consistency
grouped_100_pct_hpo = (
    equal_hpo.groupby(
        [
            "predicted_dataset",
            "automl_system",
            "method"
        ]
    )
    .agg(
        {
            "hold_out_f1_score": "mean"
        }
    )
)
grouped_100_pct_hpo = grouped_100_pct_hpo.reset_index()
grouped_100_pct_hpo["percentage"] = 100
grouped_100_pct_hpo

,predicted_dataset,automl_system,method,hold_out_f1_score,percentage
0,ed_01,Random Forest with Random Search,gmm_random,0.348548,100
1,ed_02,Random Forest with Random Search,gmm_random,0.379427,100
2,ed_03,Random Forest with Random Search,gmm_random,0.294867,100
3,ed_04,Random Forest with Random Search,gmm_random,0.904984,100
4,ed_05,Random Forest with Random Search,gmm_random,0.341667,100
5,fn_01,Random Forest with Random Search,gmm_random,0.200623,100
6,fn_02,Random Forest with Random Search,gmm_random,0.451389,100
7,fn_03,Random Forest with Random Search,gmm_random,0.198683,100
8,fn_04,Random Forest with Random Search,gmm_random,0.564875,100
9,fn_05,Random Forest with Random Search,gmm_random,0.333946,100


### Result

In [ ]:
result_equal = (
    pd.concat(
        [
            grouped_equal_concat_df,
            grouped_zero_pct_hpo,
            grouped_100_pct_hpo
            # Correct is:
            #   grouped_equal_concat_df,
            #   grouped_100_pct_hpo,
        ],
        axis=0
    )
    .sort_values(
        [
            "predicted_dataset",
            "automl_system",
            "method",
            "percentage"
        ]
    )
)
monotonic_equal = (
    result_equal.sort_values(
        [
            "predicted_dataset",
            "automl_system",
            "method",
            "percentage"
        ]
    )
    .groupby(
        [
            "predicted_dataset",
            "automl_system",
            "method"
        ],
        as_index=False,
        observed=True
    )
    ["hold_out_f1_score"]
    .apply(lambda x: x.is_monotonic_increasing)
    .rename(columns={"hold_out_f1_score": "is_monotonic_increasing"})
    [["predicted_dataset", "automl_system", "method", "is_monotonic_increasing"]]
)
result_equal = (
    result_equal.merge(
        monotonic_equal,
        on=["predicted_dataset", "automl_system", "method"],
    )
    .sort_values(["predicted_dataset", "automl_system", "method", "percentage"])
)

value_counts = result_equal["predicted_dataset"].value_counts()

assert value_counts[(value_counts > 1) & (value_counts < 4)].empty, "There should not be any dataset with 2-3 data points"

isolated = value_counts[value_counts == 1].index
assert ~grouped_equal_concat_df["predicted_dataset"].isin(isolated).any(), "There should not be any isolated datasets in concatenated datasets"
assert ~grouped_zero_pct_hpo["predicted_dataset"].isin(isolated).any(), "There should not be any isolated datasets in HPO's 0 pct"
assert (set(isolated) - set(grouped_100_pct_hpo["predicted_dataset"].unique())) == set()

result_equal = result_equal[~result_equal["predicted_dataset"].isin(isolated)]
result_equal.to_csv(
    f"/content/drive/MyDrive/Colab Notebooks/output/statistics/result_equal_monotonic_{system}_equal.csv",
    sep=";"
)
result_equal

,predicted_dataset,automl_system,method,percentage,hold_out_f1_score,is_monotonic_increasing
0,ed_01,Random Forest with Random Search,gmm_random,0,0.000000,True
1,ed_01,Random Forest with Random Search,gmm_random,25,0.000000,True
2,ed_01,Random Forest with Random Search,gmm_random,50,0.000000,True
3,ed_01,Random Forest with Random Search,gmm_random,75,0.000000,True
4,ed_01,Random Forest with Random Search,gmm_random,100,0.348548,True
...,...,...,...,...,...,...
184,ua_06,Random Forest with Random Search,gmm_random,0,0.416656,False
185,ua_06,Random Forest with Random Search,gmm_random,25,0.443624,False
186,ua_06,Random Forest with Random Search,gmm_random,50,0.443266,False
187,ua_06,Random Forest with Random Search,gmm_random,75,0.442518,False


## Diff

### 25, 50, 75

In [ ]:
grouped_diff_concat_df = (
    diff_concat_df.groupby(
        [
            "predicted_dataset",
            "automl_system",
            "method",
            "percentage"
        ]
    )
    .agg(
        {
            "hold_out_f1_score": "mean"
        }
    )
)
grouped_diff_concat_df = grouped_diff_concat_df.reset_index()
grouped_diff_concat_df

,predicted_dataset,automl_system,method,percentage,hold_out_f1_score
0,ed_01,Random Forest with Random Search,gmm_random,25,0.037344
1,ed_01,Random Forest with Random Search,gmm_random,50,0.257261
2,ed_01,Random Forest with Random Search,gmm_random,75,0.307054
3,ed_02,Random Forest with Random Search,gmm_random,25,0.000000
4,ed_02,Random Forest with Random Search,gmm_random,50,0.000000
...,...,...,...,...,...
70,ua_05,Random Forest with Random Search,gmm_random,50,0.676928
71,ua_05,Random Forest with Random Search,gmm_random,75,0.685323
72,ua_06,Random Forest with Random Search,gmm_random,25,0.431085
73,ua_06,Random Forest with Random Search,gmm_random,50,0.415511


### 0%

In [ ]:
grouped_zero_pct_hpo = (
      diff_hpo.groupby(
          [
              "predicted_dataset",
              "automl_system",
              "method"
          ]
      )
      .agg(
          {
              "hold_out_f1_score": "mean"
          }
      )
)
grouped_zero_pct_hpo = grouped_zero_pct_hpo.reset_index()
grouped_zero_pct_hpo["percentage"] = 0
grouped_zero_pct_hpo

,predicted_dataset,automl_system,method,hold_out_f1_score,percentage
0,ed_01,Random Forest with Random Search,gmm_random,0.000000,0
1,ed_02,Random Forest with Random Search,gmm_random,0.000000,0
2,fn_02,Random Forest with Random Search,gmm_random,0.304167,0
3,fn_05,Random Forest with Random Search,gmm_random,0.377322,0
4,fn_06,Random Forest with Random Search,gmm_random,0.292827,0
5,fn_07,Random Forest with Random Search,gmm_random,0.369318,0
6,fn_09,Random Forest with Random Search,gmm_random,0.261526,0
7,fn_10,Random Forest with Random Search,gmm_random,0.108333,0
8,hs_01,Random Forest with Random Search,gmm_random,0.448450,0
9,hs_02,Random Forest with Random Search,gmm_random,0.112956,0


### 100%

In [ ]:
# There is only one point per dataset since trained = predicted, but we are still grouping by and determining the mean for consistency
grouped_100_pct_hpo = (
    equal_hpo.groupby(
        [
            "predicted_dataset",
            "automl_system",
            "method"
        ]
    )
    .agg(
        {
            "hold_out_f1_score": "mean"
        }
    )
)
grouped_100_pct_hpo = grouped_100_pct_hpo.reset_index()
grouped_100_pct_hpo["percentage"] = 100
grouped_100_pct_hpo

,predicted_dataset,automl_system,method,hold_out_f1_score,percentage
0,ed_01,Random Forest with Random Search,gmm_random,0.348548,100
1,ed_02,Random Forest with Random Search,gmm_random,0.379427,100
2,ed_03,Random Forest with Random Search,gmm_random,0.294867,100
3,ed_04,Random Forest with Random Search,gmm_random,0.904984,100
4,ed_05,Random Forest with Random Search,gmm_random,0.341667,100
5,fn_01,Random Forest with Random Search,gmm_random,0.200623,100
6,fn_02,Random Forest with Random Search,gmm_random,0.451389,100
7,fn_03,Random Forest with Random Search,gmm_random,0.198683,100
8,fn_04,Random Forest with Random Search,gmm_random,0.564875,100
9,fn_05,Random Forest with Random Search,gmm_random,0.333946,100


### Result

In [ ]:
result_diff = (
    pd.concat(
        [
            grouped_diff_concat_df,
            grouped_zero_pct_hpo,
            grouped_100_pct_hpo
        ],
        axis=0
    )
    .sort_values(["predicted_dataset", "automl_system", "method", "percentage"])
)
monotonic_diff = (
    result_diff.sort_values(["predicted_dataset", "automl_system", "method", "percentage"])
    .groupby(
        [
            "predicted_dataset",
            "automl_system",
            "method"
        ],
        as_index=False, observed=True
    )
    ["hold_out_f1_score"]
    .apply(lambda x: x.is_monotonic_increasing)
    .rename(columns={"hold_out_f1_score": "is_monotonic_increasing"})
    [["predicted_dataset", "automl_system", "method", "is_monotonic_increasing"]]
)
result_diff = (
    result_diff.merge(
        monotonic_diff,
        on=["predicted_dataset", "automl_system", "method"],
    )
    .sort_values(["predicted_dataset", "automl_system", "method", "percentage"])
)

value_counts = result_diff["predicted_dataset"].value_counts()

# assert value_counts[(value_counts > 1) & (value_counts < 4)].empty, "There should not be any dataset with 2-3 data points"

isolated = value_counts[value_counts == 1].index
assert (~grouped_equal_concat_df["predicted_dataset"].isin(isolated)).any(), "There should not be any isolated datasets in concatenated datasets"
assert (~grouped_zero_pct_hpo["predicted_dataset"].isin(isolated)).any(), "There should not be any isolated datasets in HPO's 0 pct"
assert (set(isolated) - set(grouped_100_pct_hpo["predicted_dataset"].unique())) == set()

result_diff = result_diff[~result_diff["predicted_dataset"].isin(isolated)]
result_diff.to_csv(
    f"/content/drive/MyDrive/Colab Notebooks/output/statistics/result_monotonic_{system}_equal.csv",
    sep=";"
)
result_diff

,predicted_dataset,automl_system,method,percentage,hold_out_f1_score,is_monotonic_increasing
0,ed_01,Random Forest with Random Search,gmm_random,0,0.000000,True
1,ed_01,Random Forest with Random Search,gmm_random,25,0.037344,True
2,ed_01,Random Forest with Random Search,gmm_random,50,0.257261,True
3,ed_01,Random Forest with Random Search,gmm_random,75,0.307054,True
4,ed_01,Random Forest with Random Search,gmm_random,100,0.348548,True
...,...,...,...,...,...,...
154,ua_06,Random Forest with Random Search,gmm_random,0,0.416656,False
155,ua_06,Random Forest with Random Search,gmm_random,25,0.431085,False
156,ua_06,Random Forest with Random Search,gmm_random,50,0.415511,False
157,ua_06,Random Forest with Random Search,gmm_random,75,0.413447,False


# Analysis Examples

Some examples of interesting analysis below:


## Monotonic Analysis

### Equal count

In [ ]:
is_monotonic_increasing_equal = result_equal["is_monotonic_increasing"]

monotonic_increasing_equal = result_equal.loc[is_monotonic_increasing_equal]
non_monotonic_increasing_equal = result_equal.loc[~is_monotonic_increasing_equal]

print(f"Number EQUAL Monotonic increasing: {len(monotonic_increasing_equal.drop_duplicates('predicted_dataset'))}")
print(f"{monotonic_increasing_equal['predicted_dataset'].drop_duplicates().to_list()}")

print(f"Number EQUAL Non-monotonic increasing: {len(non_monotonic_increasing_equal.drop_duplicates('predicted_dataset'))}")
print(f"{non_monotonic_increasing_equal['predicted_dataset'].drop_duplicates().to_list()}")

Number EQUAL Monotonic increasing: 19
['ed_01', 'fn_02', 'fn_06', 'fn_09', 'fn_10', 'hs_02', 'hs_03', 'hs_04', 'hs_06', 'hs_08', 'hs_09', 'hs_11', 'hs_13', 'hs_14', 'pc_03', 'pc_07', 'sd_04', 'ua_01', 'ua_05']
Number EQUAL Non-monotonic increasing: 16
['ed_02', 'fn_05', 'fn_07', 'hs_01', 'hs_10', 'hs_12', 'pc_01', 'pc_02', 'pc_04', 'pc_05', 'pc_06', 'pc_08', 'sd_03', 'ua_03', 'ua_04', 'ua_06']


### Diff count

In [ ]:
is_monotonic_increasing_diff = result_diff["is_monotonic_increasing"]

monotonic_increasing_diff = result_diff.loc[is_monotonic_increasing_diff]
non_monotonic_increasing_diff = result_diff.loc[~is_monotonic_increasing_diff]

print(f"Number diff Monotonic increasing: {len(monotonic_increasing_diff.drop_duplicates('predicted_dataset'))}")
print(f"{monotonic_increasing_diff['predicted_dataset'].drop_duplicates().to_list()}")

print(f"Number diff Non-monotonic increasing: {len(non_monotonic_increasing_diff.drop_duplicates('predicted_dataset'))}")
print(f"{non_monotonic_increasing_diff['predicted_dataset'].drop_duplicates().to_list()}")

Number diff Monotonic increasing: 11
['ed_01', 'ed_02', 'fn_09', 'fn_10', 'hs_01', 'hs_04', 'hs_08', 'hs_11', 'hs_14', 'sd_03', 'sd_04']
Number diff Non-monotonic increasing: 24
['fn_02', 'fn_05', 'fn_06', 'fn_07', 'hs_02', 'hs_03', 'hs_06', 'hs_09', 'hs_10', 'hs_12', 'hs_13', 'pc_01', 'pc_02', 'pc_03', 'pc_04', 'pc_05', 'pc_06', 'pc_07', 'pc_08', 'ua_01', 'ua_03', 'ua_04', 'ua_05', 'ua_06']


### Non-monotonic equal f1_score difference

In [ ]:
(
    pd.concat(
        [
            non_monotonic_increasing_equal,
            (
                non_monotonic_increasing_equal[["hold_out_f1_score", "predicted_dataset"]]
                .shift(1)
                .rename(
                    columns={
                        "hold_out_f1_score": "hold_out_f1_score_next",
                        "predicted_dataset": "predicted_dataset_next"
                    }
                )
            )
        ],
        axis=1
    )
    .query("predicted_dataset == predicted_dataset_next")
    .assign(diff=lambda x: x["hold_out_f1_score"] - x["hold_out_f1_score_next"])
    [["predicted_dataset", "automl_system", "method", "percentage", "hold_out_f1_score", "diff"]]
)

,predicted_dataset,automl_system,method,percentage,hold_out_f1_score,diff
6,ed_02,Random Forest with Random Search,gmm_random,25,0.425741,0.425741
7,ed_02,Random Forest with Random Search,gmm_random,50,0.442578,0.016837
8,ed_02,Random Forest with Random Search,gmm_random,75,0.438371,-0.004207
9,ed_02,Random Forest with Random Search,gmm_random,100,0.379427,-0.058944
22,fn_05,Random Forest with Random Search,gmm_random,25,0.333946,-0.043376
...,...,...,...,...,...,...
178,ua_04,Random Forest with Random Search,gmm_random,100,0.334867,0.083437
185,ua_06,Random Forest with Random Search,gmm_random,25,0.443624,0.026968
186,ua_06,Random Forest with Random Search,gmm_random,50,0.443266,-0.000358
187,ua_06,Random Forest with Random Search,gmm_random,75,0.442518,-0.000748


In [ ]:
len((
    pd.concat(
        [
            non_monotonic_increasing_equal,
            (
                non_monotonic_increasing_equal[["hold_out_f1_score", "predicted_dataset"]]
                .shift(1)
                .rename(
                    columns={
                        "hold_out_f1_score": "hold_out_f1_score_next",
                        "predicted_dataset": "predicted_dataset_next"
                    }
                )
            )
        ],
        axis=1
    )
    .query("predicted_dataset == predicted_dataset_next")
    .assign(diff=lambda x: x["hold_out_f1_score"] - x["hold_out_f1_score_next"])
    [["predicted_dataset", "automl_system", "method", "percentage", "hold_out_f1_score", "diff"]]
))

64

### Non-monotonic diff f1_score difference

In [ ]:
(
    pd.concat(
        [
            non_monotonic_increasing_diff,
            (
                non_monotonic_increasing_diff[["hold_out_f1_score", "predicted_dataset"]]
                .shift(1)
                .rename(
                    columns={
                        "hold_out_f1_score": "hold_out_f1_score_next",
                        "predicted_dataset": "predicted_dataset_next"
                    }
                )
            )
        ],
        axis=1
    )
    .query("predicted_dataset == predicted_dataset_next")
    .assign(diff=lambda x: x["hold_out_f1_score"] - x["hold_out_f1_score_next"])
    [["predicted_dataset", "automl_system", "method", "percentage", "hold_out_f1_score", "diff"]]
)

,predicted_dataset,automl_system,method,percentage,hold_out_f1_score,diff
15,fn_02,Random Forest with Random Search,gmm_random,25,0.260417,-0.043750
16,fn_02,Random Forest with Random Search,gmm_random,50,0.390625,0.130208
17,fn_02,Random Forest with Random Search,gmm_random,75,0.520833,0.130208
18,fn_02,Random Forest with Random Search,gmm_random,100,0.451389,-0.069444
22,fn_05,Random Forest with Random Search,gmm_random,100,0.333946,-0.043376
...,...,...,...,...,...,...
153,ua_05,Random Forest with Random Search,gmm_random,100,0.776119,0.090796
155,ua_06,Random Forest with Random Search,gmm_random,25,0.431085,0.014429
156,ua_06,Random Forest with Random Search,gmm_random,50,0.415511,-0.015574
157,ua_06,Random Forest with Random Search,gmm_random,75,0.413447,-0.002064


In [ ]:
da_analysis = result_diff[result_diff["percentage"].isin([0, 100])] # or result_equal since it has the same effect
da_analysis = da_analysis.merge(da_analysis, on=["predicted_dataset", "automl_system", "method"])
da_analysis = da_analysis[
    (da_analysis["percentage_x"] != da_analysis["percentage_y"])
]
da_analysis = da_analysis.drop_duplicates(["predicted_dataset", "automl_system", "method"])
da_analysis["diff"] = da_analysis["hold_out_f1_score_y"] - da_analysis["hold_out_f1_score_x"]
da_analysis.sort_values("diff", ignore_index=True)

,predicted_dataset,automl_system,method,percentage_x,hold_out_f1_score_x,is_monotonic_increasing_x,percentage_y,hold_out_f1_score_y,is_monotonic_increasing_y,diff
0,fn_05,Random Forest with Random Search,gmm_random,0,0.377322,False,100,0.333946,False,-0.043376
1,pc_05,Random Forest with Random Search,gmm_random,0,0.394640,False,100,0.372681,False,-0.021959
2,hs_10,Random Forest with Random Search,gmm_random,0,0.566667,False,100,0.550000,False,-0.016667
3,fn_07,Random Forest with Random Search,gmm_random,0,0.369318,False,100,0.366477,False,-0.002841
4,hs_14,Random Forest with Random Search,gmm_random,0,0.517544,True,100,0.517544,True,0.000000
5,hs_04,Random Forest with Random Search,gmm_random,0,0.906004,True,100,0.906004,True,0.000000
6,hs_06,Random Forest with Random Search,gmm_random,0,0.571293,False,100,0.577408,False,0.006114
7,ua_06,Random Forest with Random Search,gmm_random,0,0.416656,False,100,0.442095,False,0.025439
8,ua_01,Random Forest with Random Search,gmm_random,0,0.520599,False,100,0.558380,False,0.037781
9,pc_02,Random Forest with Random Search,gmm_random,0,0.445113,False,100,0.498584,False,0.053470
